In [2]:
from typing import List, Union, Dict, Optional

Token = Union[int, str]  # int для чисел, str для операторов/скобок/переменных


def tokenize(expr: str, allow_vars: bool = False) -> List[Token]:
    """
    Превращает строку в список токенов:
    - числа (int)
    - операторы: + - * /
    - скобки: ( )
    - (опционально) переменные: 'a'..'z'
    
    Поддержка унарного минуса:
    - "-3" -> число -3
    - "-(2+3)" -> преобразуем в "0 - (2+3)"
    """
    tokens: List[Token] = []
    i = 0
    n = len(expr)

    def prev_is_op_or_lparen() -> bool:
        """Унарный минус возможен в начале или после оператора/скобки '('."""
        if not tokens:
            return True
        prev = tokens[-1]
        return prev in ("+", "-", "*", "/", "(")

    while i < n:
        ch = expr[i]

        # пропускаем пробелы
        if ch.isspace():
            i += 1
            continue

        # число
        if ch.isdigit():
            j = i
            while j < n and expr[j].isdigit():
                j += 1
            tokens.append(int(expr[i:j]))
            i = j
            continue

        # переменная (задача 3*)
        if allow_vars and ("a" <= ch <= "z"):
            tokens.append(ch)
            i += 1
            continue

        # скобки
        if ch in ("(", ")"):
            tokens.append(ch)
            i += 1
            continue

        # операторы
        if ch in ("+", "-", "*", "/"):
            # обработка унарного минуса
            if ch == "-" and prev_is_op_or_lparen():
                # если дальше число -> сразу читаем отрицательное число
                if i + 1 < n and expr[i + 1].isdigit():
                    j = i + 1
                    while j < n and expr[j].isdigit():
                        j += 1
                    tokens.append(-int(expr[i + 1:j]))
                    i = j
                    continue
                # если дальше '(' -> делаем "0 - ( ... )"
                if i + 1 < n and expr[i + 1] == "(":
                    tokens.append(0)
                    tokens.append("-")
                    i += 1
                    continue

            tokens.append(ch)
            i += 1
            continue

        # прочие символы (по условию выражение корректное, но на всякий случай)
        raise ValueError(f"Недопустимый символ: {ch!r}")

    return tokens


def to_rpn(tokens: List[Token]) -> List[Token]:
    """
    Перевод токенов в обратную польскую нотацию (RPN) по Shunting Yard.
    """
    out: List[Token] = []
    ops: List[str] = []

    prec = {"+": 1, "-": 1, "*": 2, "/": 2}

    for t in tokens:
        if isinstance(t, int) or (isinstance(t, str) and len(t) == 1 and t.isalpha()):
            # число или переменная -> сразу в выход
            out.append(t)
            continue

        if t in prec:
            # оператор: вытаскиваем из стека операторы с >= приоритетом
            while ops and ops[-1] in prec and prec[ops[-1]] >= prec[t]:
                out.append(ops.pop())
            ops.append(t)
            continue

        if t == "(":
            ops.append(t)
            continue

        if t == ")":
            while ops and ops[-1] != "(":
                out.append(ops.pop())
            ops.pop()  # убрать '('
            continue

        raise ValueError(f"Неожиданный токен: {t!r}")

    while ops:
        out.append(ops.pop())

    return out


def eval_rpn(rpn: List[Token], vars_map: Optional[Dict[str, int]] = None) -> int:
    """
    Вычисление RPN стеком.
    Деление '/' делаем как целочисленное с усечением к нулю: int(a / b).
    """
    if vars_map is None:
        vars_map = {}

    st: List[int] = []

    for t in rpn:
        if isinstance(t, int):
            st.append(t)
            continue

        if isinstance(t, str) and len(t) == 1 and t.isalpha():
            if t not in vars_map:
                raise ValueError(f"Нет значения переменной: {t}")
            st.append(int(vars_map[t]))
            continue

        # оператор
        b = st.pop()
        a = st.pop()

        if t == "+":
            st.append(a + b)
        elif t == "-":
            st.append(a - b)
        elif t == "*":
            st.append(a * b)
        elif t == "/":
            # ВАЖНО: Python '//' округляет вниз, поэтому используем int(a / b)
            st.append(int(a / b))
        else:
            raise ValueError(f"Неизвестный оператор: {t}")

    return st[-1]


def eval_expression(expr: str) -> int:
    """Задача 2: вычислить выражение (без переменных)."""
    tokens = tokenize(expr, allow_vars=False)
    rpn = to_rpn(tokens)
    return eval_rpn(rpn)


# Примеры:
tests = [
    "1 + 2*3",
    "(1 + 2) * 3",
    "10/3",
    "-3 + 5*2",
    "-(2+3)*4",
    "7 + 8/(-3)",
]

for t in tests:
    print(t, "=", eval_expression(t))

1 + 2*3 = 7
(1 + 2) * 3 = 9
10/3 = 3
-3 + 5*2 = 7
-(2+3)*4 = -20
7 + 8/(-3) = 5


In [5]:
def eval_expression_with_vars(expr: str) -> int:
    """
    1) находим переменные (a..z)
    2) запрашиваем значения через input()
    3) вычисляем выражение
    """
    tokens = tokenize(expr, allow_vars=True)

    # найти уникальные переменные
    vars_set = sorted({t for t in tokens if isinstance(t, str) and len(t) == 1 and t.isalpha()})

    vars_map: Dict[str, int] = {}
    for v in vars_set:
        while True:
            try:
                vars_map[v] = int(input(f"Введите значение {v}: ").strip())
                break
            except ValueError:
                print("Нужно целое число. Повторите ввод.")

    rpn = to_rpn(tokens)
    return eval_rpn(rpn, vars_map)


# Пример:
expr = "a*(b+3) - c/2"
print("Результат =", eval_expression_with_vars(expr))

Результат = 740591
